# Legal Contract Ticket Decision Engine
# [Model from Eng:Baraa Drive](https://drive.google.com/drive/folders/1PvGcKPLTUDeeu6bWS-qRuhfc8VffgaYd?usp=sharing)


## Overview
This assignment trains you to think like a **real system**, not like a lawyer and not like a chatbot.

Your system does **one job only**:
> Decide the **next action** for a user request related to **legal contract issues**.

The system **does NOT**:
- Give legal advice
- Explain contract clauses
- Suggest what the user should do

It only decides **what happens next**.

---

## Allowed Actions (Strict)
Your output **must be one of these five actions only**:

```
CREATE_TICKET
ASK_FOR_MORE_INFO
REJECT_REQUEST
FLAG_OUT_OF_SCOPE
ESCALATE_TO_HUMAN
```

Any other action = ❌ **Fail**

---

## Action Definitions

### 1. CREATE_TICKET
Use this action **only if all are true**:
- The issue is legal
- The issue is related to a contract
- A real dispute is described
- Information is enough to open a ticket

To create a ticket, the message must clearly indicate:
- A written agreement or contract
- At least two parties
- A specific dispute or legal event
If any of these are missing, do NOT create a ticket.

Example:
> "The tenant left before the end date and refused to pay the penalty."

---

### 2. ASK_FOR_MORE_INFO
Use this action if:
- The issue is clearly contract-related
- But important details are missing

Example:
> "There is a problem with the agreement."

---

### 3. REJECT_REQUEST
Use this action if the user:
- Asks for legal advice
- Asks to explain a contract clause
- Asks what they should do

Example:
> "Is this penalty clause fair?"

---

### 4. FLAG_OUT_OF_SCOPE
Use this action if the request is:
- Not legal
- Legal but **not related to contracts**
- Personal or general complaints

Example:
> "Someone threatened me yesterday."

---

### 5. ESCALATE_TO_HUMAN
Use this action if:
- The issue is contract-related
- The situation is very sensitive or risky
- The system should not decide alone

Example:
> "The partner used a loophole in the agreement to take all profits."

---

## Input Format
You receive **only one input**:

```json
{
  "user_message": "string"
}
```

No extra fields. You must understand everything from the text.

---

## Output Format (Flexible)


You are **NOT required** to use a specific output structure or JSON format.

You may present your output in **any format you prefer**, such as:
- JSON
- Table
- Bullet points

### What matters is **NOT the shape**, but the **decision**.

Your output must clearly show:
1. **The chosen action** (one of the allowed actions)
2. **Your confidence level** (any reasonable scale is acceptable)
3. **A short explanation** of why this action was selected

### Example (one possible format)

Action: ASK_FOR_MORE_INFO  
Confidence: Medium  
Reasoning: The message suggests a contract-related dispute, but key details are missing.
#### Or
```json
{
  "action": "ONE_OF_THE_FIVE_ACTIONS",
  "confidence": 0.0,
  "reasoning": "Short, neutral explanation"
}
```
### Important Notes
- Do **not** give legal advice
- Do **not** explain laws or contract clauses
- Focus only on deciding the **next system action**
- Keep reasoning short and neutral



You are free to design the output format in the way that best helps you explain your thinking.

----


# Notes:

---
## 1) What Does a Good Answer Look Like?

A good answer is **not** about choosing the “perfect” action.

A good answer:
- Respects the system boundaries
- Chooses one of the allowed actions
- Explains the decision clearly and logically

If your reasoning is strong and consistent,  
you may receive full credit even if another action could also be reasonable.

---
## 2) Real-World Note

In real companies, systems like this fail not because they are “not smart enough”,
but because they act when they should not.

This assignment trains you to build systems that:
- Know their limits
- Make safe decisions
- Protect users and the company

---
## 3) How This Assignment Is Structured

This assignment is divided into **three parts**, each one building on the previous one.

Do **not** treat the parts as separate tasks.  
They represent **progressive levels of system thinking**:

- First, you learn where the system **must stop**
- Then, you decide when the system **can act**
- Finally, you handle **uncertainty and risk**

Read each part carefully before starting.  
The same rules and allowed actions apply to **all parts**.



In [ ]:
!pip install bitsandbytes accelerate
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from google.colab import drive
import torch
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path =  "/content/drive/MyDrive/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
      quantization_config=bnb_config,
  torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded locally from Drive


## Quick Check Task — Model Run

### Goal
Confirm that the model is running and responding correctly.


### Instructions
1. Ask the model one simple question.
2. Make sure it returns a clear answer.
3. Copy the question and the response.


In [ ]:
prompt  = "explain supervised learning in simple terms"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)
inputs

{'input_ids': tensor([[ 5649,  2428, 11292,  6509,   297,  2560,  4958]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
input_token_num =  inputs["input_ids"].shape[1]
input_token_num

7

In [ ]:
with torch.no_grad():
  outputs  = model.generate(
      **inputs,
      max_new_tokens=512, #
      temperature=0,
      do_sample=False # الالتزام بالبرومبت الي بتدخله اياه
      )

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
Fullanswer = tokenizer.decode(outputs, skip_special_tokens=True)
Fullanswer[0]

"explain supervised learning in simple terms.\n\nSupervised learning is a type of machine learning where the algorithm learns from labeled data. In this process, the algorithm is provided with input data along with the correct output. The algorithm uses this data to learn a function that maps inputs to outputs. The goal is to create a model that can accurately predict the output for new, unseen data based on the patterns it learned from the training data.\n\nHere's a simple analogy to understand supervised learning:\n\nImagine you're teaching a child to recognize different types of fruits. You show them pictures of apples, bananas, and oranges, each labeled with the correct name. The child observes the pictures and learns to associate each image with its corresponding name. Over time, the child becomes better at identifying the fruit in the picture by learning the visual patterns and features that distinguish each type of fruit.\n\nIn supervised learning, the algorithm is like the chil

In [ ]:
gen_tokens = outputs[0][input_token_num:]

In [ ]:
answer = tokenizer.decode(gen_tokens, skip_special_tokens=True)
print(answer)

.

Supervised learning is a type of machine learning where the algorithm learns from labeled data. In this process, the algorithm is provided with input data along with the correct output. The algorithm uses this data to learn a function that maps inputs to outputs. The goal is to create a model that can accurately predict the output for new, unseen data based on the patterns it learned from the training data.

Here's a simple analogy to understand supervised learning:

Imagine you're teaching a child to recognize different types of fruits. You show them pictures of apples, bananas, and oranges, each labeled with the correct name. The child observes the pictures and learns to associate each image with its corresponding name. Over time, the child becomes better at identifying the fruit in the picture by learning the visual patterns and features that distinguish each type of fruit.

In supervised learning, the algorithm is like the child, and the labeled data is like the pictures with na

# Generation Function

In [ ]:

# =====================================================
# 2) SYSTEM PROMPTS
# =====================================================
DECISION_SYSTEM_PROMPT = (
    "You are a Contract Issue Routing System.\n"
    "Your job is to decide the next system action based ONLY on a user's message related to legal contract issues.\n\n"

    "You are NOT a lawyer.\n"
    "You do NOT provide legal advice.\n"
    "You do NOT explain legal concepts.\n"
    "You do NOT suggest what the user should do.\n"
    "You ONLY select the correct system action.\n\n"

    "AVAILABLE ACTIONS (ONLY THESE ARE ALLOWED):\n"
    "- CREATE_TICKET\n"
    "- ASK_FOR_MORE_INFO\n"
    "- REJECT_REQUEST\n"
    "- FLAG_OUT_OF_SCOPE\n"
    "- ESCALATE_TO_HUMAN\n\n"

    "You MUST choose EXACTLY ONE action.\n\n"

    "--------------------------------------------------\n"
    "MANDATORY DECISION PRIORITY ORDER\n"
    "You MUST evaluate actions in this EXACT order and stop once a condition is satisfied.\n\n"

    "STEP 1 — REJECT_REQUEST\n"
    "Select REJECT_REQUEST if the user asks for legal advice, legal interpretation, fairness judgment,\n"
    "or asks what action they should take.\n\n"

    "STEP 2 — FLAG_OUT_OF_SCOPE\n"
    "Select FLAG_OUT_OF_SCOPE if the message does not describe a legal issue related to a contract.\n"
    "This includes situations involving threats, harassment, criminal behavior,\n"
    "personal conflicts, or general complaints that are not clearly tied to a contract.\n\n"

    "STEP 3 — ASK_FOR_MORE_INFO\n"
    "Select ASK_FOR_MORE_INFO if the message appears related to a contract,\n"
    "but essential information is missing.\n"
    "Essential information includes confirmation of a contract, identification of parties,\n"
    "a specific violation, or a concrete event.\n\n"

    "STEP 4 — CREATE_TICKET\n"
    "Select CREATE_TICKET ONLY if ALL of the following are clearly present:\n"
    "- A written agreement or contract is explicitly stated or clearly implied\n"
    "- At least two identifiable parties are involved\n"
    "- A specific breach, violation, or dispute is described\n"
    "- A concrete event has already occurred\n"
    "- The information provided is sufficient to open a case without assumptions\n\n"

    "You MUST NOT assume a contract exists.\n"
    "If contract existence is unclear, you MUST select ASK_FOR_MORE_INFO instead.\n\n"

    "STEP 5 — ESCALATE_TO_HUMAN\n"
    "Select ESCALATE_TO_HUMAN only if a contract clearly exists, a specific dispute is present,\n"
    "and the situation involves severe harm, exploitation, extreme complexity, or significant risk\n"
    "that requires human judgment beyond automated routing.\n\n"

    "--------------------------------------------------\n"
    "INPUT FORMAT:\n"
    "You will receive:\n"
    "{ \"user_message\": \"text\" }\n\n"

    "You MUST base your decision ONLY on the provided message.\n"
    "Do NOT assume missing facts.\n"
    "Do NOT invent information.\n"
    "Do NOT infer contracts without explicit or clearly implied evidence.\n\n"

    "--------------------------------------------------\n"
    "OUTPUT REQUIREMENTS:\n"
    "You MUST return EXACTLY this JSON format:\n\n"

    "{\n"
    "  \"action\": \"CREATE_TICKET | ASK_FOR_MORE_INFO | REJECT_REQUEST | FLAG_OUT_OF_SCOPE | ESCALATE_TO_HUMAN\",\n"
    "  \"confidence\": <number between 0.00 and 1.00>,\n"
    "  \"reasoning\": \"short neutral factual explanation\"\n"
    "}\n\n"

    "--------------------------------------------------\n"
    "CRITICAL OUTPUT RULES:\n"
    "- Output MUST be valid JSON\n"
    "- Do NOT output plain text\n"
    "- Do NOT output markdown\n"
    "- Do NOT include extra fields\n"
    "- Do NOT include extra commentary\n"
    "- Do NOT provide legal advice\n\n"

    "You are a strict decision engine. Your only responsibility is selecting the correct action.\n"
)

EXPLANATION_SYSTEM_PROMPT = (
    "You are a customer communication assistant.\n"
    "Your job is to explain the system result to the user clearly, politely, and professionally.\n\n"

    "IMPORTANT RULES:\n"
    "- Do NOT mention internal system rules.\n"
    "- Do NOT mention decision categories or classifications.\n"
    "- Do NOT mention words like CREATE_TICKET, REJECT_REQUEST, etc.\n"
    "- Do NOT provide legal advice.\n"
    "- Do NOT explain contract law.\n\n"

    "IF the request was accepted or moved forward:\n"
    "- Inform the user that their issue has been received.\n"
    "- Briefly summarize the issue in simple language.\n"
    "- Keep the tone professional and neutral.\n\n"

    "IF more information is required:\n"
    "- Clearly state what information is missing.\n"
    "- Ask for the missing details politely.\n"
    "- Keep it short and specific.\n\n"

    "IF the request was rejected:\n"
    "- State the reason in ONE clear bullet point only.\n"
    "- Do NOT add extra commentary.\n\n"

    "STYLE REQUIREMENTS:\n"
    "- Keep the explanation short.\n"
    "- Use simple and user-friendly language.\n"
    "- Avoid legal terminology.\n"
    "- Be calm, neutral, and respectful.\n\n"

    "Output must be plain text only.\n"
    "Do NOT return JSON.\n"
    "Do NOT include markdown formatting.\n"
)

In [ ]:
import json
import re
import torch

def generate_text(
    prompt,
    tokenizer,
    model,
    device,
    do_sample=False,
    temperature=0.0,
    max_new_tokens=128,
    seed=42
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature
        )

    gen_tokens = outputs[0][input_tokens:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return text

In [ ]:
def extract_json_only(text: str) -> dict:
    # حذف ```json ``` إذا موجودة
    text = text.replace("```json", "").replace("```", "")

    # استخراج أول JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON found")

    return json.loads(match.group(0))

In [ ]:
def userPrompt(user_input):
    decision_messages = [
        {"role": "system", "content": DECISION_SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    decision_prompt = tokenizer.apply_chat_template(
        decision_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    decision = generate_text(
        decision_prompt,
        tokenizer,
        model,
        device,
        do_sample=False,
        temperature=0.0
    )
    print("***"*50, "\n")
    print("\n🤖 SYSTEM DECISION (generate_text output):\n", decision)
    llm_output_json = extract_json_only(decision)
    print("***"*50 , "\n")
    print("\n🤖 SYSTEM DECISION (generate_text output as JSON):\n", llm_output_json)
    print("***"*50, "\n")
    return llm_output_json

In [ ]:
def generate_user_explanation(result_json, tokenizer, model, device):

    explain_messages = [
        {"role": "system", "content": EXPLANATION_SYSTEM_PROMPT},
        {"role": "user", "content": f"System result: {result_json}"}
    ]

    explain_prompt = tokenizer.apply_chat_template(
        explain_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    final_answer = generate_text(
        explain_prompt,
        tokenizer,
        model,
        device,
        do_sample=True,       # هنا مسموح الإبداع
        temperature=0.7
    )

    return final_answer


In [ ]:
# =====================================================
# 3) FUNCTIONS (System Side – NOT the model)
# =====================================================

def create_support_ticket(llm_output_json):
    print("🎫 [SYSTEM Decision] Creating support ticket... \n (For example: store in DataBase)")
    # message generate_user_explanation from aonther llm
    message = generate_user_explanation(llm_output_json, tokenizer, model, device)
    print("***"*50)
    print("🎫 [SYSTEM user explanation]...")
    print(message)


def ask_for_more_info(llm_output_json):
    print("🎫 [SYSTEM Decision] We Ask for more info... \n (For example: new textbox for user)")
    message = generate_user_explanation(llm_output_json, tokenizer, model, device)
    print("***"*50)
    print("🎫 [SYSTEM user explanation]...")
    print(message)


def reject_request(llm_output_json):
    print("🎫 [SYSTEM Decision] Reject request...  (For example: warning for 3 times and block after that) ")
    message = generate_user_explanation(llm_output_json, tokenizer, model, device)
    print("***"*50)
    print("🎫 [SYSTEM user explanation]...")
    print(message)

def flag_out_of_scope(llm_output_json):
    print("🎫 [SYSTEM Decision] Flag Out Of Scope... (The question you are asking is out of the scope of the system) ")
    message = generate_user_explanation(llm_output_json, tokenizer, model, device)
    print("***"*50)
    print("🎫 [SYSTEM user explanation]...")
    print(message)

def escelate_to_human(llm_output_json):
    print("🎫 [SYSTEM Decision] You will be directed to human to handle your special request ")
    message = generate_user_explanation(llm_output_json, tokenizer, model, device)
    print("***"*50)
    print("🎫 [SYSTEM user explanation]...")
    print(message)

def tool_calling(llm_output_json):
    print("System Side (Not LLM) \n")
    if llm_output_json["action"] == "CREATE_TICKET":
        decision_json = create_support_ticket(llm_output_json) # tool DB , server ,
    elif llm_output_json["action"] == "ASK_FOR_MORE_INFO":
        decision_json = ask_for_more_info(llm_output_json) # send to llm ag
    elif llm_output_json["action"] == "REJECT_REQUEST":
        decision_json = reject_request(llm_output_json) # close
    elif llm_output_json["action"] == "FLAG_OUT_OF_SCOPE":
        decision_json = flag_out_of_scope(llm_output_json)
    elif llm_output_json["action"] == "ESCELATE_TO_HUMAN":
        decision_json = escelate_to_human(llm_output_json)
    else:
        result = {"message": "Unknown action"}


---

# Part 1 — Scope Control (Beginner)

### Goal
Learn when the system **should not act**.

### Task
For each message, choose the correct action.

| User Message | Expected Action |
|-------------|----------------|
| "I want to know my rights" | REJECT_REQUEST |
| "There is a dispute with a company" | ASK_FOR_MORE_INFO |
| "I was threatened" | FLAG_OUT_OF_SCOPE |

### Deliverable
- Your decision rules (text)
- 10 examples written by you

---


In [ ]:
llm_output_json = userPrompt("I signed a one-year lease agreement with my landlord, but he is now demanding that I leave the apartment after only six months without any valid reason.")
tool_calling(llm_output_json)

****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output):
 {
  "action": "CREATE_TICKET",
  "confidence": 0.95,
  "reasoning": "The user has provided a written agreement (one-year lease), identifiable parties (the user and the landlord), a specific breach (demand to leave without valid reason), and a concrete event (the landlord's demand)."
}
****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output as JSON):
 {'action': 'CREATE_TICKET', 'confidence': 0.95, 'reasoning': "The user has provided a written agreement (one-year lease), identifiable parties (the user and the landlord), a specific breach (demand to leave without valid reason), and a concrete event (the landlord's demand)."}
**********************************

In [ ]:
llm_output_json = userPrompt("I want to know my rights") #####################
tool_calling(llm_output_json)

****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output):
 {
  "action": "ASK_FOR_MORE_INFO",
  "confidence": 0.85,
  "reasoning": "The user is inquiring about rights, which suggests a potential contract issue, but lacks specific details about a contract or a particular violation."
}
****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output as JSON):
 {'action': 'ASK_FOR_MORE_INFO', 'confidence': 0.85, 'reasoning': 'The user is inquiring about rights, which suggests a potential contract issue, but lacks specific details about a contract or a particular violation.'}
****************************************************************************************************************************************************** 

S

In [ ]:
llm_output_json = userPrompt("There is a dispute with a company")
tool_calling(llm_output_json)

****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output):
 {
  "action": "ASK_FOR_MORE_INFO",
  "confidence": 0.85,
  "reasoning": "The message indicates a dispute with a company, which suggests a potential contract issue. However, essential details such as the nature of the dispute, the parties involved, and specific breaches are missing."
}
Here, ASK_FOR_MORE_INFO is chosen because the user has mentioned a dispute, which implies a contract issue, but the message lacks specific details necessary to proceed further.
****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output as JSON):
 {'action': 'ASK_FOR_MORE_INFO', 'confidence': 0.85, 'reasoning': 'The message indicates a dispute with a company, which suggests a pote

In [ ]:
llm_output_json = userPrompt("My client has not paid me yet.")
tool_calling(llm_output_json)

****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output):
 {
  "action": "CREATE_TICKET",
  "confidence": 0.95,
  "reasoning": "The user's message indicates a potential breach of contract due to non-payment, which involves a written agreement (implied by the existence of a client-service provider relationship), at least two identifiable parties (the client and the service provider), and a specific event (non-payment)."
}
****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output as JSON):
 {'action': 'CREATE_TICKET', 'confidence': 0.95, 'reasoning': "The user's message indicates a potential breach of contract due to non-payment, which involves a written agreement (implied by the existence of a client-service provider 

In [ ]:
llm_output_json = userPrompt("I was threatened.")
tool_calling(llm_output_json)

****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output):
 {
  "action": "REJECT_REQUEST",
  "confidence": 1.00,
  "reasoning": "The user's message indicates a threat, which is outside the scope of contract-related issues and does not pertain to a legal contract dispute."
}
****************************************************************************************************************************************************** 


🤖 SYSTEM DECISION (generate_text output as JSON):
 {'action': 'REJECT_REQUEST', 'confidence': 1.0, 'reasoning': "The user's message indicates a threat, which is outside the scope of contract-related issues and does not pertain to a legal contract dispute."}
****************************************************************************************************************************************************** 

System Side (Not LLM) 


# Part 2 — Information Judgment (Intermediate)

### Goal
Decide if the information is **enough** to create a ticket.

### Task
- Choose an action
- Explain your reasoning briefly

Example:

```json
{
  "action": "ASK_FOR_MORE_INFO",
  "confidence": 0.62,
  "reasoning": "A contract dispute is implied, but contract type and parties are missing."
}
```

### Deliverable
- A checklist for CREATE_TICKET
- 10 difficult examples

---



# Part 3 — Ambiguity & Risk (Advanced)

### Goal
Handle unclear and risky situations.

### Rules
- The word "contract" may NOT appear in the message

Examples:

> "The company ended the cooperation suddenly despite our written agreement"

Expected:
- ASK_FOR_MORE_INFO **or** ESCALATE_TO_HUMAN

> "Can this clause force me to pay a penalty?"

Expected:
- REJECT_REQUEST

### Deliverable
- 10 ambiguous messages
- Chosen action for each
- Why other actions were NOT chosen

---



# Part 4 (Optional) — Decision Maker API using FastAPI



## Why This Task Matters

In real companies, LLMs are **not used as notebooks or demos**.
They are deployed as **services** that other systems depend on.

Product teams, backend services, and automation pipelines
cannot "talk" to a chatbot.
They need **stable, predictable decisions** exposed through APIs.

This task simulates how LLMs are actually used in production:

* As a decision layer
* Behind an HTTP interface
* With strict contracts and validation

This is the same pattern used in:

* Ticketing systems
* Internal automation tools
* Agent-based platforms
* AI-powered workflows in companies

---

## Task Type

**Optional (Advanced / Production-Oriented)**

This task is for students who want to:

* Think like production engineers
* Understand real-world AI system design
* Go beyond notebooks into deployable systems

---

## Task Goal

Convert your **LLM Decision Maker** into a real **API service**.

The LLM will no longer generate text for humans.
It will:

* Receive a request
* Decide what action to take
* Return a strict machine-readable response

---

## What You Will Build

You will build a **FastAPI service** that exposes a single endpoint
used as a **Decision Layer**.

This API represents how companies integrate LLMs into systems.

---

## API Specification

### Endpoint

```
POST /decide
```

### Input (JSON)

```json
{
  "user_request": "I want to report a payment issue",
  "user_id": "12345"
}
```

### Output from LLM (JSON)

```json
{
  "action": "CREATE_TICKET",
  "priority": "HIGH"
}
```

No explanations.
No free text.
Only valid schema output.

---

## System Rules (Mandatory)

Your LLM must follow these rules:

* Output must match the response schema exactly
* No additional text is allowed
* Same input must produce the same output
* No guessing or assumptions
* Any schema violation is considered a failure

---

## Decision Logic Requirements

Inside the LLM decision module:

* Use closed output space
* Use deterministic decoding
* Enforce strict rules
* Validate output before returning it

The API must never trust raw LLM output.

---

## Error Handling

* If the LLM output is invalid → retry once
* If it fails again → return HTTP 400
* Do not interpret or fix invalid outputs

Fail fast, as in real systems.

---

## What NOT To Do

* Do not build a chat interface
* Do not allow free-text responses
* Do not enable sampling or temperature
* Do not bypass schema validation

---

## Deliverables

Students who choose this task must submit:

1. FastAPI project folder
2. Working `/decide` endpoint
3. README.md explaining:

   * System purpose
   * API contract
   * Decision rules

---

## Final Note

This task reflects how AI engineers work in real companies.

You are not training a model.
You are **building a system** that other systems rely on.